In [93]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn import linear_model
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from math import sqrt
df = pd.read_excel("../../clean_coast_data.xlsx")

In [94]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25898 entries, 0 to 25897
Data columns (total 32 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   coast_num           25898 non-null  int64         
 1   Area                25898 non-null  object        
 2   Distance from gaza  24095 non-null  float64       
 3   center              25898 non-null  int64         
 4   city rank           25898 non-null  int64         
 5   bathing_season      25898 non-null  int64         
 6   summer_vaction      25898 non-null  int64         
 7   winter              25898 non-null  int64         
 8   river               25898 non-null  int64         
 9   military            25898 non-null  int64         
 10  nature              25898 non-null  int64         
 11  coast_name          25898 non-null  object        
 12  coast_desc          25354 non-null  object        
 13  Date test           25898 non-null  datetime64

In [95]:
df['Area'].unique()

array(['North', 'Center', 'Eilat', 'South'], dtype=object)

In [96]:
# using get_dummies on education column
encoded = pd.get_dummies(df.Area)
# Concatenate the dummies to original dataframe
merged_data = pd.concat([df, encoded], axis=1)
# dropping the original column which was not encoded
merged_data.drop(columns='Area', inplace=True)

In [97]:
merged_data.drop(columns=['coast_name','coast_desc','city','check station','key'], inplace=True)

Quick & Dirty

In [98]:
without_nulls = merged_data.dropna()
merged_data['Date test num'] = merged_data['Date test'].view(np.int64)
print(df.shape, without_nulls.shape)
without_nulls.sort_values('Date test')
y = without_nulls[['test_result','Date test']]
X = without_nulls.drop('test_result',axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=True, test_size = 0.2)
X_train_scaler = MinMaxScaler()
X_test_scaler = MinMaxScaler()
X_train_scaled = X_train_scaler.fit_transform(X_train.drop('Date test',axis=1))
X_test_scaled = X_test_scaler.fit_transform(X_test.drop('Date test',axis=1))

mlr = linear_model.LinearRegression()
mlr.fit(X_train_scaled,y_train.test_result)

prediction = mlr.predict(X_train_scaled)
resTrain =  pd.DataFrame(
    data={
        'Prediction':prediction, 
        'Actual': y_train.test_result.values    },
        index=y_train['Date test']
)
prediction = mlr.predict(X_test_scaled)
resTest = pd.DataFrame(
    data={
        'Prediction':prediction,
        'Actual': y_test.test_result.values    },
        index=y_test['Date test']
)

print('rmse train',sqrt(mean_squared_error(resTrain.Prediction, resTrain.Actual)))
print('std train',np.std(resTrain.Actual))
print('Sample')
print(resTrain.sample(10))
print()

print('rmse test',sqrt(mean_squared_error(resTest.Prediction, resTest.Actual)))
print('std test',np.std(resTest.Actual))
print('Sample')
print(resTest.sample(10))
print()

(25898, 32) (21987, 30)
rmse train 0.9619805146251541
std train 0.9930339579551059
Sample
            Prediction  Actual
Date test                     
2016-01-26    3.924451       1
2020-11-29    3.649188       4
2018-04-30    3.672014       4
2016-01-11    3.892160       4
2022-12-31    3.595350       2
2013-03-17    3.657417       2
2005-08-17    3.735533       3
2010-12-26    3.773432       3
2017-04-25    3.721331       5
2020-11-29    4.220641       4

rmse test 0.8707274743035849
std test 0.9825529489547785
Sample
            Prediction  Actual
Date test                     
2022-06-30    3.917401       5
2005-09-19    3.245222       1
2010-09-05    3.261884       2
2018-06-12    3.508422       3
2017-04-25    3.772911       5
2014-03-09    2.939772       2
2019-08-15    3.787268       5
2013-03-04    3.086413       1
2009-06-28    3.850344       4
2011-04-13    3.081515       3



with univariant feature selection

In [101]:
X.columns

Index(['coast_num', 'Distance from gaza', 'center', 'city rank',
       'bathing_season', 'summer_vaction', 'winter', 'river', 'military',
       'nature', 'Date test', 'test_result_num', 'month', 'year',
       'city_jinny_ranking', 'hifa_port_dis', 'sore_lengh', 'temperture ',
       'rain', 'rain-1day', 'rain-2days', 'rain-3days', 'temp-1day',
       'temp-2days', 'temp avg 3 days', 'Center', 'Eilat', 'North', 'South'],
      dtype='object')

In [99]:
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_regression

uni_X = 

# f_reg_selector = SelectKBest(f_regression, k=6)
# f_reg_selector.fit(X.drop('Date test', axis=1, inplace=True), y[['test_result']])
# f_reg_support = f_reg_selector.get_support()
# f_reg_feature = X.loc[:,f_reg_support].columns.to_list()
# print("The top 6 top scored features are ",f_reg_feature)